In [1]:
import os
import sys
# Adiciona o diretório pai (raiz do projeto) ao sys.path
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np

In [3]:
import pandas as pd
import polars as pl
from preprocessing import pipeline as pipel

### Preparação dos Dados

In [4]:
dataf = pl.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_base.parquet")
df = dataf.to_pandas()

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8732 entries, 0 to 8731
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   texto   8732 non-null   str  
 1   label   8732 non-null   str  
 2   tipo    8732 non-null   str  
dtypes: str(3)
memory usage: 796.8 MB


In [6]:
import re

_INTEIRO_TEOR_PATTERNS = [
    (re.compile(r"(Erro Parser)?>>>>>inicio<<<<<\n?", re.MULTILINE | re.IGNORECASE), ""),
    (re.compile(r"fimid:\d+|#####fim#####id:\d+\n?", re.MULTILINE), ""),
]
_MULTI_SPACE = re.compile(r" {2,}")

def limpar_inteiro_teor(text: str) -> str:
    for pattern, replacement in _INTEIRO_TEOR_PATTERNS:
        text = pattern.sub(replacement, text)
    return _MULTI_SPACE.sub(" ", text).strip()


df["texto"] = df["texto"].dropna().map(limpar_inteiro_teor)

### Estratégia de Truncamento

### Configuração do Treino do Modelo

In [7]:
import sys
import torch
import transformers
import sentence_transformers
from transformers import TrainingArguments

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

c:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
Torch: 2.12.0+cpu
Transformers: 4.57.6
Sentence Transformers: 5.5.1


In [8]:
#model_name = "dominguesm/legal-bert-base-cased-ptbr"
model_name = 'neuralmind/bert-base-portuguese-cased'

In [9]:
from datasets import Dataset
import losses

In [10]:
dataset_total = Dataset.from_pandas(df)
dataset_split = dataset_total.train_test_split(test_size=0.2, seed=42)
dataset_split

DatasetDict({
    train: Dataset({
        features: ['texto', 'label', 'tipo'],
        num_rows: 6985
    })
    test: Dataset({
        features: ['texto', 'label', 'tipo'],
        num_rows: 1747
    })
})

In [12]:
from datasets import load_dataset
from sentence_transformers.losses import CosineSimilarityLoss

from setfit import SetFitModel, SetFitTrainer, sample_dataset


# Load a dataset from the Hugging Face Hub
dataset = dataset_split
num_classes = df['label'].nunique()

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    use_differentiable_head=True,
    head_params={"out_features": num_classes},
)

# Create trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    loss_class=losses.SupConLoss,
    metric="accuracy",
    batch_size=8,
    num_iterations=2, # The number of text pairs to generate for contrastive learning
    num_epochs=2, # The number of epochs to use for contrastive learning
    column_mapping={"texto": "text", "label": "label"} # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.freeze() # Freeze the head
trainer.train() # Train only the body

# Unfreeze the head and freeze the body -> head-only training
trainer.unfreeze(keep_body_frozen=True)
# or
# Unfreeze the head and unfreeze the body -> end-to-end training
# trainer.unfreeze(keep_body_frozen=False)

trainer.train(
    num_epochs=25, # The number of epochs to train the head or the whole model (body and head)
    batch_size=8,
    body_learning_rate=1e-5, # The body's learning rate
    learning_rate=1e-2, # The head's learning rate
    l2_weight=0.0, # Weight decay on **both** the body and head. If `None`, will use 0.01.
)
metrics = trainer.evaluate()
model.save_pretrained("modelos/bertimbau-supcon-model")


# Run inference
# preds = model(["i loved the spiderman movie!", "pineapple on pizza is the worst 🤮"])

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
C:\Users\lfmelo\AppData\Local\Temp\ipykernel_11316\2216700222.py:19: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
Map: 100%|██████████| 6985/6985 [00:08<00:00, 784.33 examples/s] 
C:\Users\lfmelo\AppData\Local\Temp\ipykernel_11316\2216700222.py:32: DeprecationWarning: `SetFitTrainer.freeze` is deprecated and will be removed in v2.0.0 of SetFit. Please use `SetFitModel.freeze` directly instead.
  trainer.freeze() # Freeze the head


MemoryError: 

In [ ]:
temp = 0.3 
criterion = losses.SupConLoss(temperature=temp)

train_sample = sample_dataset(
    dataset_split["train"],
    label_column="label",
    num_samples=3
)

model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    use_differentiable_head=True,
    head_params={"out_features": num_classes},
)

trainer = SetFitTrainer(
    model=model,
    train_dataset=train_sample,
    eval_dataset=None,
    loss_class=CosineSimilarityLoss,
    metric="accuracy",
    batch_size=2,
    num_iterations=1,
    num_epochs=1,
    column_mapping={"texto": "text", "label": "label"}
)

#trainer.freeze()
trainer.train()

c:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lfmelo\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\lfmelo\Documents\Github\TJGO_ThemeC

Step,Training Loss
1,0.022300
50,0.160700
100,0.156200
150,0.226600


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

# 1. Load high-dimensional data (1797 samples, 64 features each)
digits = load_digits()
X = digits.data
y = digits.target

# 2. Initialize and configure the t-SNE model
tsne = TSNE(
    n_components=2,  # Target dimension (2D for plotting)
    perplexity=30.0,  # Number of nearest neighbors to consider
    learning_rate="auto",  # Automatically chooses learning rate
    init="pca",  # Initialize embedding using PCA for stability
    random_state=42,  # Set random state for reproducibility
)

# 3. Fit and transform the data
X_embedded = tsne.fit_transform(X)

# 4. Plot the 2D representation using Seaborn
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=X_embedded[:, 0],
    y=X_embedded[:, 1],
    hue=y,
    palette=sns.color_palette("hls", 10),
    legend="full",
    alpha=0.8,
)

# Add titles and labels (Note: axes values themselves are non-interpretable)
plt.title("t-SNE Visualization of Handwritten Digits", fontsize=16)
plt.xlabel("t-SNE Component 1", fontsize=12)
plt.ylabel("t-SNE Component 2", fontsize=12)
plt.legend(title="Digit Class")
plt.show()
